# YOLO Predicted ROI + roi_repr Grad-CAM 端到端解释

在 GBCU 测试集上可视化完整 pipeline：

```
Full USG image
  → YOLO11 detector (multiple boxes)
  → crop each predicted ROI
  → roi_repr classifier (AE / CLS)
  → per-ROI Grad-CAM
  → fuse ROI predictions → image-level diagnosis
```

每张图输出一张解释图，自动标注：
- YOLO box confidence
- ROI classifier prediction & malignant probability
- image-level fusion result
- GT label & TP/TN/FP/FN case type

默认优先 **AE**（binary task 最佳），并支持 **5 方法同样本对比**（cls / ae / vae / siamese / triplet）× **5 张 test 样本**。

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from typing import List, Sequence

import matplotlib.pyplot as plt
import torch
from IPython.display import Image as IPyImage, display
from PIL import Image

from roi_repr.config import ReprConfig
from roi_repr.data.splits import build_cv_folds, build_roi_transform
from roi_repr.eval_multi_roi import load_model_from_checkpoint
from roi_repr.eval_yolo_roi_pipeline import load_yolo_model
from roi_repr.gradcam import (
    ALL_METHODS,
    RoiGradCAM,
    binary_case_type,
    build_overlay_rgb,
    compare_methods_on_sample,
    explain_image,
    render_explanation_figure,
    render_multi_method_comparison_figure,
    run_explanation_batch,
    run_multi_method_comparison_batch,
    save_explanation_figure,
    save_multi_method_comparison_figure,
)

# ========= 路径与实验开关 =========
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "GBCU").is_dir() and (PROJECT_ROOT.parent / "GBCU").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / "GBCU"
YOLO_CKPT = PROJECT_ROOT / "outputs/yolov11_onestep_run1/weights/best.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs/yolo_roi_gradcam"

# 主方法：ae（推荐）或 cls（对比基线）
METHOD = "ae"
ALL_COMPARE_METHODS: Sequence[str] = ALL_METHODS  # ae(左), cls, vae, siamese, triplet
N_COMPARE_SAMPLES = 5
# 五方法对比用的 test 样本：None=取前 N_COMPARE_SAMPLES 张；或指定文件名列表
COMPARE_IMAGE_NAMES: List[str] | None = None  # 例如 ["im00335.jpg", "im00224.jpg", ...]
TASK_MODE = "binary"
FOLD = 0

# YOLO 推理（与 eval_yolo_roi_pipeline 一致）
EVAL_SCORE_THRESHOLD = 0.5
NMS_IOU_THRESHOLD = 0.5
MAX_DETECTIONS = 5
IMGSZ = 800
NO_BOX_FALLBACK = "full_image"

# 批量可视化控制
MAX_IMAGES = 12          # None = 全部 test 集
CASE_FILTER = None       # 例如 ("FP", "FN") 只看错例；None = 全部
GRADCAM_TARGET = "pred"  # "pred" 或 "malignant"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATASET_ROOT = {DATASET_ROOT} (exists={DATASET_ROOT.is_dir()})")
print(f"YOLO_CKPT exists = {YOLO_CKPT.is_file()}")
print(f"device = {DEVICE}")

In [ ]:
def load_pipeline(method: str, fold: int):
    cfg = ReprConfig(method=method, task_mode=TASK_MODE)
    cfg.dataset_root = DATASET_ROOT
    cfg.__post_init__()

    ckpt = cfg.output_dir / f"fold{fold}" / "checkpoints" / "best_model.pth"
    if not ckpt.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt}")

    classifier = load_model_from_checkpoint(cfg, ckpt, DEVICE)
    transform = build_roi_transform(cfg)
    gradcam = RoiGradCAM(classifier)
    gradcam.attach()
    return cfg, classifier, transform, gradcam, ckpt


cfg, classifier, transform, gradcam, classifier_ckpt = load_pipeline(METHOD, FOLD)
yolo_model = load_yolo_model(YOLO_CKPT)
folds = build_cv_folds(cfg)
test_samples = folds[FOLD]["test"]
imgs_dir = cfg.dataset_root / "imgs"

print(f"method={cfg.method} fold={FOLD} task={cfg.task_mode}")
print(f"classifier_ckpt = {classifier_ckpt}")
print(f"test N = {len(test_samples)}")

## 单张图像调试

修改 `DEMO_IMAGE` 可快速查看任意 test 样本的完整解释图。

In [ ]:
DEMO_IMAGE = test_samples[0][0]  # 例如 "im00335.jpg"

demo_path = imgs_dir / DEMO_IMAGE
demo_gt = next(cls for name, cls in test_samples if name == DEMO_IMAGE)
demo_gt_mapped = cfg.map_label(demo_gt)

image = Image.open(demo_path).convert("RGB")
record = explain_image(
    image,
    DEMO_IMAGE,
    demo_gt_mapped,
    image_path=demo_path,
    yolo_model=yolo_model,
    classifier=classifier,
    gradcam=gradcam,
    transform=transform,
    cfg=cfg,
    device=DEVICE,
    eval_score_threshold=EVAL_SCORE_THRESHOLD,
    nms_iou_threshold=NMS_IOU_THRESHOLD,
    max_detections=MAX_DETECTIONS,
    imgsz=IMGSZ,
    no_box_fallback=NO_BOX_FALLBACK,
    gradcam_target=GRADCAM_TARGET,
)

print(f"image={DEMO_IMAGE}")
print(f"GT={cfg.class_tag(record['gt_class'])}  Fused={cfg.class_tag(record['pred_class'])}  case={record['case_type']}")
for i, roi in enumerate(record["roi_details"]):
    print(
        f"  ROI#{i+1}: conf={roi.get('conf', 0):.3f} pred={cfg.class_tag(roi['pred'])} "
        f"P(mal)={roi['mal_prob']:.3f} source={roi.get('source', 'yolo')}"
    )

demo_out = OUTPUT_DIR / METHOD / f"fold{FOLD}" / "demo" / f"{Path(DEMO_IMAGE).stem}_{record['case_type']}.jpg"
save_explanation_figure(image, record, demo_out, cfg, method=METHOD, fold=FOLD)
display(IPyImage(filename=str(demo_out)))

## AE Grad-CAM + GT 标注框对比（仅预览，不保存）

并排查看 **3 张** test 图像的 **AE** Grad-CAM，并叠加 `bbox_annot.json` 中放射科医生标注：

| 图层 | 含义 |
|------|------|
| 热力图 | AE classifier Grad-CAM（YOLO 预测 ROI 上） |
| 红色实线 | YOLO 预测框（pipeline 实际 crop 区域） |
| 青色虚线 | GT `nml` / `abn`（胆囊 ROI 区域） |
| 黄色实线 | GT `malg`（恶性病灶区域） |

（不显示结石 `stn` 与良性壁增厚 `bmt`）

用于直观判断：模型关注区域是否与临床标注的病灶/胆囊区域对齐。

In [ ]:
import importlib
import roi_repr.gradcam as gradcam_module

importlib.reload(gradcam_module)
from train_resnet50_roi_classifier import load_bbox_annotations
from roi_repr.gradcam import get_gt_boxes_for_image, render_pred_gt_gradcam_preview

# 仅叠加胆囊 ROI (nml/abn) + 恶性病灶 (malg)；不含 stn/bmt
GT_FOCUS_LABELS = frozenset({"nml", "abn", "malg"})

N_GT_PREVIEW = 3
# 指定 3 张图名；None 则取 test 集前 N_GT_PREVIEW 张
GT_PREVIEW_IMAGES: List[str] | None = None  # 例如 ["im00757.jpg", "im00335.jpg", "im01085.jpg"]

ae_cfg, ae_model, ae_transform, ae_gradcam, _ = load_pipeline("ae", FOLD)
bbox_annot = load_bbox_annotations(DATASET_ROOT / "bbox_annot.json")
preview_names = GT_PREVIEW_IMAGES or [name for name, _ in test_samples[:N_GT_PREVIEW]]

preview_triplets = []
try:
    ae_gradcam.attach()
    for image_name in preview_names:
        image_path = imgs_dir / image_name
        raw_cls = next(cls for name, cls in test_samples if name == image_name)
        gt_mapped = ae_cfg.map_label(raw_cls)
        image = Image.open(image_path).convert("RGB")
        gt_boxes = get_gt_boxes_for_image(
            bbox_annot,
            image_name,
            include_labels=GT_FOCUS_LABELS,
        )

        record = explain_image(
            image,
            image_name,
            gt_mapped,
            image_path=image_path,
            yolo_model=yolo_model,
            classifier=ae_model,
            gradcam=ae_gradcam,
            transform=ae_transform,
            cfg=ae_cfg,
            device=DEVICE,
            eval_score_threshold=EVAL_SCORE_THRESHOLD,
            nms_iou_threshold=NMS_IOU_THRESHOLD,
            max_detections=MAX_DETECTIONS,
            imgsz=IMGSZ,
            no_box_fallback=NO_BOX_FALLBACK,
            gradcam_target=GRADCAM_TARGET,
        )
        preview_triplets.append((image, record, gt_boxes))
        gt_box_tags = [g["label"] for g in gt_boxes]
        print(
            f"{image_name}: annot={gt_box_tags} | "
            f"img_GT={ae_cfg.class_tag(record['gt_class'])} | "
            f"fused={ae_cfg.class_tag(record['pred_class'])} [{record['case_type']}]"
        )

    fig = render_pred_gt_gradcam_preview(preview_triplets, ae_cfg, method="ae", fold=FOLD)
    plt.show()
finally:
    ae_gradcam.detach()

## 批量生成 test 集解释图

In [ ]:
batch_out = OUTPUT_DIR / METHOD / f"fold{FOLD}" / "vis"
records = run_explanation_batch(
    test_samples,
    cfg=cfg,
    classifier=classifier,
    yolo_model=yolo_model,
    device=DEVICE,
    output_dir=batch_out,
    fold=FOLD,
    imgs_dir=imgs_dir,
    max_images=MAX_IMAGES,
    case_filter=CASE_FILTER,
    eval_score_threshold=EVAL_SCORE_THRESHOLD,
    nms_iou_threshold=NMS_IOU_THRESHOLD,
    max_detections=MAX_DETECTIONS,
    imgsz=IMGSZ,
    no_box_fallback=NO_BOX_FALLBACK,
    gradcam_target=GRADCAM_TARGET,
)

case_counts = Counter(r["case_type"] for r in records)
print(f"Saved {len(records)} figures -> {batch_out}")
print("case distribution:", dict(case_counts))

preview_n = min(4, len(records))
fig, axes = plt.subplots(1, preview_n, figsize=(5 * preview_n, 5))
if preview_n == 1:
    axes = [axes]
for ax, rec in zip(axes, records[:preview_n]):
    ax.imshow(Image.open(rec["vis_path"]))
    ax.set_title(f"{rec['image_name']} [{rec['case_type']}]")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 五方法同样本 Grad-CAM 对比

对 **5 张 test 样本**，每张跑 **5 种方法**（**AE 在最左列**）。

**输出目录结构**（`outputs/yolo_roi_gradcam/compare_5methods/fold{N}/`）：

```
compare_5methods/fold0/
├── panels/                          # 25 张单图（5样本 × 5方法）
│   ├── im00335_ae.jpg
│   ├── im00335_cls.jpg
│   ├── im00335_vae.jpg
│   ├── ...
└── summary_5x5_grid.jpg             # 1 张 5×5 总览大图（notebook 内展示）
```

- `panels/`：每张为完整解释图（含 ROI 缩略图、conf、P(mal) 等）
- `summary_5x5_grid.jpg`：notebook 中一张整图便于观察对比

In [ ]:
# 选取 5 个 test 样本
if COMPARE_IMAGE_NAMES:
    compare_samples = [(n, next(c for name, c in test_samples if name == n)) for n in COMPARE_IMAGE_NAMES]
else:
    compare_samples = test_samples[:N_COMPARE_SAMPLES]

print(f"5-method comparison on {len(compare_samples)} samples (AE = leftmost column):")
for name, raw in compare_samples:
    print(f"  {name}  raw_label={raw}  binary={cfg.class_tag(cfg.map_label(raw))}")


def make_method_loader(fold: int):
    """缓存 classifier 权重，每次调用返回新的 RoiGradCAM（由 compare 函数负责 attach/detach）。"""
    cache: dict = {}

    def _loader(method: str):
        if method not in cache:
            m_cfg = ReprConfig(method=method, task_mode=TASK_MODE)
            m_cfg.dataset_root = DATASET_ROOT
            m_cfg.__post_init__()
            ckpt = m_cfg.output_dir / f"fold{fold}" / "checkpoints" / "best_model.pth"
            if not ckpt.is_file():
                raise FileNotFoundError(f"Checkpoint not found: {ckpt}")
            m_model = load_model_from_checkpoint(m_cfg, ckpt, DEVICE)
            m_transform = build_roi_transform(m_cfg)
            cache[method] = (m_cfg, m_model, m_transform)
        m_cfg, m_model, m_transform = cache[method]
        return m_cfg, m_model, m_transform, RoiGradCAM(m_model)

    return _loader


compare_out_dir = OUTPUT_DIR / "compare_5methods" / f"fold{FOLD}"
compare_batch = run_multi_method_comparison_batch(
    compare_samples,
    ALL_COMPARE_METHODS,
    loader=make_method_loader(FOLD),
    yolo_model=yolo_model,
    device=DEVICE,
    imgs_dir=imgs_dir,
    output_dir=compare_out_dir,
    fold=FOLD,
    map_label_fn=cfg.map_label,
    class_tag_fn=cfg.class_tag,
    max_samples=N_COMPARE_SAMPLES,
    eval_score_threshold=EVAL_SCORE_THRESHOLD,
    nms_iou_threshold=NMS_IOU_THRESHOLD,
    max_detections=MAX_DETECTIONS,
    imgsz=IMGSZ,
    no_box_fallback=NO_BOX_FALLBACK,
    gradcam_target=GRADCAM_TARGET,
)

multi_compare_results = compare_batch["results"]
print(f"\nSaved {compare_batch['n_panels']} panel images -> {compare_batch['panels_dir']}")
print(f"Saved summary grid -> {compare_batch['summary_grid_path']}")
print(f"Method order (left→right): {' | '.join(ALL_COMPARE_METHODS)}")

for item in multi_compare_results:
    cases = {m: item["method_records"][m]["case_type"] for m in ALL_COMPARE_METHODS}
    print(f"  {item['image_name']}: {cases}")

# notebook 中展示一张 5×5 总览大图
summary_path = Path(compare_batch["summary_grid_path"])
display(IPyImage(filename=str(summary_path)))

## 按 case type 抽样浏览

从已生成的解释图中，每种 TP/TN/FP/FN 各展示一张（若存在）。

In [ ]:
import numpy as np

by_case = {case: None for case in ("TP", "TN", "FP", "FN")}
for rec in records:
    case = rec["case_type"]
    if by_case[case] is None:
        by_case[case] = rec

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, case in zip(axes.flatten(), ("TP", "TN", "FP", "FN")):
    rec = by_case[case]
    if rec is None:
        ax.text(0.5, 0.5, f"No {case} in batch", ha="center", va="center")
        ax.set_title(case)
        ax.axis("off")
        continue
    ax.imshow(Image.open(rec["vis_path"]))
    ax.set_title(f"{case}: {rec['image_name']}")
    ax.axis("off")
plt.suptitle(f"Case-type gallery ({METHOD.upper()}, fold={FOLD})", fontsize=13)
plt.tight_layout()
plt.show()